# Human-in-the-Loop (HITL) 패턴 - 솔루션

## 학습 목표
- Human-in-the-Loop 패턴을 `interrupt_before`로 구현할 수 있다
- `MemorySaver` 체크포인터와 함께 그래프를 중단/재개할 수 있다
- `graph.update_state()`로 사람의 결정을 그래프 상태에 반영할 수 있다
- 실전 예제에서 코드 리뷰 에이전트를 구현할 수 있다

## 참조 자료
- 학습 문서: `docs/part5-advanced/11-quality-assurance.md` (섹션 2.5)
- LangGraph HITL: https://langchain-ai.github.io/langgraph/concepts/human_in_the_loop/

## 개념 요약

### interrupt_before 작동 원리

```
[generate] → [review] ──interrupt_before──→ [execute]
                                                ↑
                                    사람이 확인 후
                             graph.invoke(None, config) 로 재개
```

### 핵심 API

| API | 설명 |
|-----|------|
| `graph.compile(checkpointer=MemorySaver(), interrupt_before=["node"])` | 특정 노드 실행 전 중단 설정 |
| `graph.invoke(input, config)` | 그래프 실행 (중단 지점까지) |
| `graph.update_state(config, {"key": value})` | 중단된 상태에 사람의 결정 반영 |
| `graph.invoke(None, config)` | 중단된 지점부터 재개 |

### MemorySaver의 역할

```
MemorySaver = 체크포인터
  → 그래프가 중단될 때 현재 상태를 메모리에 저장
  → thread_id로 상태를 식별
  → 재개 시 저장된 상태에서 이어서 실행
```

In [ ]:
import sys
sys.path.insert(0, '../..')

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from common.fake_llm import FakeLLM

print("Human-in-the-Loop 솔루션 준비 완료")

## 실습 1 솔루션: interrupt_before 기본 패턴

In [ ]:
class ApprovalState(TypedDict):
    task: str
    plan: str | None
    approved: bool | None
    result: str | None

plan_llm = FakeLLM(responses={
    "default": "1. 데이터베이스 백업\n2. 스키마 변경 적용\n3. 애플리케이션 재시작"
})

def generate_node(state: ApprovalState) -> dict:
    """실행 계획을 생성합니다."""
    plan = plan_llm.invoke(state["task"]).content
    print(f"생성된 계획:\n{plan}")
    return {"plan": plan}

def review_node(state: ApprovalState) -> dict:
    """생성된 계획을 출력합니다. (interrupt_before 전 마지막 노드)"""
    print(f"[검토 대기 중] 계획: {state['plan']}")
    return state

def execute_node(state: ApprovalState) -> dict:
    """승인된 계획을 실행합니다."""
    result = f"실행 완료: {state['plan']}"
    print(f"실행 결과: {result}")
    return {"result": result}

In [ ]:
def build_approval_graph():
    """interrupt_before 기본 패턴 그래프를 구성합니다."""
    graph = StateGraph(ApprovalState)
    
    graph.add_node("generate", generate_node)
    graph.add_node("review", review_node)
    graph.add_node("execute", execute_node)
    
    graph.set_entry_point("generate")
    graph.add_edge("generate", "review")
    graph.add_edge("review", "execute")
    graph.add_edge("execute", END)
    
    return graph.compile(
        checkpointer=MemorySaver(),
        interrupt_before=["execute"],
    )

app1 = build_approval_graph()
print("그래프 구성 완료")

In [ ]:
config1 = {"configurable": {"thread_id": "approval-001"}}

# Step 1: 그래프 실행 (execute 노드 전에 중단됨)
print("=== Step 1: 실행 (execute 전 중단) ===")
state_after_interrupt = app1.invoke(
    {
        "task": "프로덕션 DB 스키마 변경",
        "plan": None,
        "approved": None,
        "result": None,
    },
    config=config1,
)
print(f"중단 후 상태: plan={state_after_interrupt.get('plan', 'N/A')[:30]}...")

# Step 2: 사람이 계획을 확인하고 재개 결정
print("\n--- 사람이 계획을 검토합니다 ---")
print("계획이 적절합니다. 실행을 승인합니다.")

# Step 3: 중단된 지점부터 재개
print("\n=== Step 3: 재개 ===")
result1 = app1.invoke(None, config=config1)
print(f"최종 결과: {result1.get('result', 'N/A')}")

In [ ]:
# 검증 셀
assert result1 is not None, "결과가 None입니다"
assert result1.get("plan") is not None, "plan이 생성되어야 합니다"
assert result1.get("result") is not None, "execute 후 result가 있어야 합니다"
assert "실행 완료" in result1["result"], f"result에 '실행 완료'가 포함되어야 합니다. 현재: {result1['result']}"
print("최종 결과:", result1["result"])
print("✅ 실습 1 완료! interrupt_before로 중단 후 재개가 성공했습니다.")

## 실습 2 솔루션: 승인/거부 분기

In [ ]:
class DecisionState(TypedDict):
    task: str
    plan: str | None
    approved: bool | None
    result: str | None
    attempt_count: int

def make_plan_node(state: DecisionState) -> dict:
    """계획을 생성합니다. attempt_count를 증가시킵니다."""
    plan = plan_llm.invoke(state["task"]).content
    new_count = state.get("attempt_count", 0) + 1
    print(f"계획 생성 (시도 {new_count}회): {plan[:40]}...")
    return {"plan": plan, "attempt_count": new_count}

def approval_gate_node(state: DecisionState) -> dict:
    """승인 게이트 (패스스루 - interrupt_before로 중단됨)."""
    print(f"[승인 대기] 계획: {state.get('plan', 'N/A')[:40]}...")
    return state

def check_decision(state: DecisionState) -> str:
    """승인 여부에 따라 분기합니다."""
    if state.get("approved") is True:
        return "execute"
    return "regenerate"

def execute_approved_node(state: DecisionState) -> dict:
    """승인된 계획을 실행합니다."""
    result = f"[승인됨] 실행 완료: {state['plan']}"
    print(result)
    return {"result": result}

def regenerate_node(state: DecisionState) -> dict:
    """거부 시 재생성을 알립니다."""
    result = f"[거부됨] 재생성이 필요합니다. 시도 {state.get('attempt_count', 0)}회"
    print(result)
    return {"result": result}

In [ ]:
def build_decision_graph():
    """승인/거부 분기 그래프를 구성합니다."""
    graph = StateGraph(DecisionState)
    
    graph.add_node("make_plan", make_plan_node)
    graph.add_node("approval_gate", approval_gate_node)
    graph.add_node("execute_approved", execute_approved_node)
    graph.add_node("regenerate", regenerate_node)
    
    graph.set_entry_point("make_plan")
    graph.add_edge("make_plan", "approval_gate")
    graph.add_conditional_edges(
        "approval_gate",
        check_decision,
        {"execute": "execute_approved", "regenerate": "regenerate"},
    )
    graph.add_edge("execute_approved", END)
    graph.add_edge("regenerate", END)
    
    return graph.compile(
        checkpointer=MemorySaver(),
        interrupt_before=["approval_gate"],
    )

app2 = build_decision_graph()
print("승인/거부 분기 그래프 구성 완료")

In [ ]:
# 시나리오 A: 승인
config_approve = {"configurable": {"thread_id": "decision-approve"}}

# Step 1: 실행
print("=== 시나리오 A: 승인 ===")
app2.invoke(
    {"task": "서버 점검", "plan": None, "approved": None, "result": None, "attempt_count": 0},
    config=config_approve,
)

# Step 2: 사람이 승인
print("\n--- 사람이 승인합니다 ---")
app2.update_state(config_approve, {"approved": True})

# Step 3: 재개
result_approve = app2.invoke(None, config=config_approve)
print("승인 시나리오 결과:", result_approve["result"])

In [ ]:
# 시나리오 B: 거부
config_reject = {"configurable": {"thread_id": "decision-reject"}}

# Step 1: 실행
print("=== 시나리오 B: 거부 ===")
app2.invoke(
    {"task": "위험한 배포", "plan": None, "approved": None, "result": None, "attempt_count": 0},
    config=config_reject,
)

# Step 2: 사람이 거부
print("\n--- 사람이 거부합니다 ---")
app2.update_state(config_reject, {"approved": False})

# Step 3: 재개
result_reject = app2.invoke(None, config=config_reject)
print("거부 시나리오 결과:", result_reject["result"])

In [ ]:
# 검증 셀
assert result_approve is not None, "승인 결과가 None입니다"
assert result_reject is not None, "거부 결과가 None입니다"
assert "승인" in result_approve["result"], f"승인 결과에 '승인됨'이 포함되어야 합니다. 현재: {result_approve['result']}"
assert "거부" in result_reject["result"], f"거부 결과에 '거부됨'이 포함되어야 합니다. 현재: {result_reject['result']}"
print("승인 결과:", result_approve["result"])
print("거부 결과:", result_reject["result"])
print("✅ 실습 2 완료! update_state로 승인/거부 분기가 올바르게 작동합니다.")

## 실습 3 솔루션: 실전 예제 - 코드 리뷰 에이전트

In [ ]:
class CodeReviewState(TypedDict):
    requirement: str
    code: str | None
    review_feedback: str | None
    approved: bool | None
    deploy_result: str | None
    revision_count: int

code_gen_llm = FakeLLM(responses={
    "수정": "def login(user, password):\n    # 피드백 반영: 입력 검증 추가\n    if not user or not password:\n        raise ValueError('입력값이 없습니다')\n    return authenticate(user, password)",
    "default": "def login(user, password):\n    return authenticate(user, password)"
})

def generate_code_node(state: CodeReviewState) -> dict:
    """요구사항에 따라 코드를 생성합니다."""
    code = code_gen_llm.invoke(state["requirement"]).content
    print(f"생성된 코드:\n{code}")
    return {"code": code, "revision_count": 0}

def human_review_node(state: CodeReviewState) -> dict:
    """인간 리뷰 게이트 (interrupt_before로 중단됨)."""
    print(f"\n[리뷰 대기] 코드 (수정 {state.get('revision_count', 0)}회):\n{state.get('code', 'N/A')}")
    return state

def route_decision(state: CodeReviewState) -> str:
    """승인 여부에 따라 분기합니다."""
    if state.get("approved") is True:
        return "deploy"
    return "fix_code"

def deploy_node(state: CodeReviewState) -> dict:
    """코드를 배포합니다."""
    revision_count = state.get("revision_count", 0)
    code = state.get("code", "")
    deploy_result = f"배포 완료 (수정 {revision_count}회)\n{code}"
    print(f"\n{deploy_result}")
    return {"deploy_result": deploy_result}

def fix_code_node(state: CodeReviewState) -> dict:
    """피드백을 반영하여 코드를 수정합니다."""
    feedback = state.get("review_feedback", "")
    new_code = code_gen_llm.invoke(f"수정 요청: {feedback}").content
    new_revision_count = state.get("revision_count", 0) + 1
    print(f"\n피드백 반영 후 수정된 코드 (수정 {new_revision_count}회):\n{new_code}")
    return {
        "code": new_code,
        "revision_count": new_revision_count,
        "approved": None,  # 다음 리뷰를 위해 초기화
    }

In [ ]:
def build_code_review_graph():
    """코드 리뷰 HITL 에이전트 그래프를 구성합니다."""
    graph = StateGraph(CodeReviewState)
    
    graph.add_node("generate_code", generate_code_node)
    graph.add_node("human_review", human_review_node)
    graph.add_node("deploy", deploy_node)
    graph.add_node("fix_code", fix_code_node)
    
    graph.set_entry_point("generate_code")
    graph.add_edge("generate_code", "human_review")
    graph.add_conditional_edges(
        "human_review",
        route_decision,
        {"deploy": "deploy", "fix_code": "fix_code"},
    )
    graph.add_edge("fix_code", "human_review")  # 순환 엣지
    graph.add_edge("deploy", END)
    
    return graph.compile(
        checkpointer=MemorySaver(),
        interrupt_before=["human_review"],
    )

app3 = build_code_review_graph()
print("코드 리뷰 에이전트 그래프 구성 완료")

In [ ]:
# 시나리오: 거부 → 수정 → 승인
config3 = {"configurable": {"thread_id": "code-review-001"}}

initial_state = {
    "requirement": "로그인 함수 구현",
    "code": None,
    "review_feedback": None,
    "approved": None,
    "deploy_result": None,
    "revision_count": 0,
}

# Step 1: 코드 생성 (human_review 전 중단)
print("=== Step 1: 코드 생성 ===")
app3.invoke(initial_state, config=config3)

# Step 2: 1차 리뷰 - 거부
print("\n=== Step 2: 1차 리뷰 - 거부 ===")
app3.update_state(config3, {
    "approved": False,
    "review_feedback": "입력 검증이 없습니다"
})

# Step 3: 코드 수정 후 2차 리뷰 전 중단
print("\n=== Step 3: 코드 수정 후 2차 리뷰 ===")
app3.invoke(None, config=config3)

# Step 4: 2차 리뷰 - 승인
print("\n=== Step 4: 2차 리뷰 - 승인 ===")
app3.update_state(config3, {
    "approved": True,
    "review_feedback": None,
})

# Step 5: 배포
print("\n=== Step 5: 배포 ===")
result3 = app3.invoke(None, config=config3)

In [ ]:
# 검증 셀
assert result3 is not None, "최종 결과가 None입니다"
assert result3.get("deploy_result") is not None, "배포 결과가 없습니다"
assert "배포 완료" in result3["deploy_result"], f"배포 완료 메시지가 없습니다. 현재: {result3['deploy_result']}"
assert result3.get("revision_count", 0) >= 1, "코드 수정이 최소 1회 이루어져야 합니다"
print("최종 배포 결과:")
print(result3["deploy_result"])
print(f"\n총 수정 횟수: {result3['revision_count']}회")
print("✅ 실습 3 완료! 코드 리뷰 HITL 에이전트가 올바르게 작동합니다.")

## 정리

이번 실습에서 배운 내용:

| 개념 | 설명 | 사용법 |
|------|------|--------|
| `MemorySaver` | 그래프 상태를 메모리에 저장하는 체크포인터 | `compile(checkpointer=MemorySaver())` |
| `interrupt_before` | 특정 노드 실행 전 그래프 중단 | `compile(interrupt_before=["node_name"])` |
| `invoke(None, config)` | 중단된 지점부터 재개 | `graph.invoke(None, config)` |
| `update_state` | 중단된 상태에 사람의 결정 반영 | `graph.update_state(config, {"key": value})` |
| `thread_id` | 여러 실행을 구분하는 식별자 | `{"configurable": {"thread_id": "..."}}`  |

### HITL 패턴의 핵심

```
1. compile(interrupt_before=["gate_node"]) 로 중단 지점 지정
2. invoke(initial_state, config) 로 중단까지 실행
3. update_state(config, {"approved": True/False}) 로 결정 반영
4. invoke(None, config) 로 재개
```

## 다음 단계
- Module 12: 멀티 에이전트 시스템